___
# <center>Aula 5: o pipeline e a gramática de gráficos</center>
___

## Aula 05

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * escrever uma análise como uma sequência de operações encadeadas;
 * usar os cinco verbos do pandas, e dizer por que a ordem importa;
 * definir um gráfico como um mapeamento de variáveis em aspectos estéticos;
 * montar um gráfico camada por camada, no plotnine;
 * explicar a diferença entre pintar de uma cor e mapear uma variável em cor.

Este notebook é o **espelho dos slides**: as seções seguem a ordem do que está
projetado, e o código é o mesmo. Ele serve para você acompanhar rodando, e
depois para reler em casa.

A última seção é a **Gincana do Pipeline**, com o enunciado de cada rodada e uma
célula em branco para você escrever a resposta. Os gabaritos não estão aqui:
eles aparecem no telão depois que a rodada fecha.

> O que vai além do slide (histograma, densidade, boxplot, facetas, rótulos e
> exercícios) está no notebook **aula05_extra_graficos**, para estudar por conta.


___
<div id="indice"></div>

## Índice

- [A tabela de hoje](#dados)

- [A pergunta de hoje](#pergunta)

- [Encadear operações](#encadear)
    - [Do jeito da aula 3](#antigo)
    - [🔗 A mesma coisa, encadeada](#encadeado)
    - [Os cinco verbos](#verbos)
    - [A ordem importa](#ordem)

- [O que é um gráfico](#definicao)

- [A gramática de gráficos](#gramatica)

- [Uma camada de cada vez](#camadas)

- [🎨 Pintar não é mapear](#pintar)

- [A regra vale para toda estética](#regra-aes)

- [Geometrias e tipos de variáveis](#geometrias)

- [A Gincana do Pipeline](#gincana)

- [Quinta-feira](#depois)


___
<div id="dados"></div>

# A tabela de hoje

A base de apelações criminais do TJSP, a mesma da aula 4.


O plotnine não vem instalado no Google Colab. Tire o `#` da segunda linha, rode
uma vez, e ponha o `#` de volta:


In [ ]:
# no Colab, rode uma vez:
# %pip install -q plotnine


In [ ]:
import pandas as pd
from plotnine import *

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


In [ ]:
criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

criminal["regime"] = pd.Categorical(
    criminal["regime_inicial"],
    categories=["aberto", "semiaberto", "fechado"],
    ordered=True,
)

penas = (
    criminal
    .dropna(subset=["regime", "pena_anos"])
    .query("pena_anos <= 30")
)

criminal.shape, penas.shape


São duas tabelas, e as duas aparecem nas cartas da gincana:

* **`criminal`** é a base inteira, com 475 acórdãos;
* **`penas`** é a mesma base com duas decisões já tomadas: fora quem não tem
  regime ou pena, e fora as penas acima de 30 anos. Ela tem uma coluna a mais,
  `regime`, que é a versão **ordenada** de `regime_inicial`.

`.head()` mostra as primeiras linhas. É a primeira coisa a fazer depois de ler
um arquivo: se a tabela veio torta, você descobre agora e não daqui a vinte
células.


In [ ]:
criminal.head()


In [ ]:
penas.head()


[Volta ao Índice](#indice)


___
<div id="pergunta"></div>

# A pergunta de hoje

> Nas apelações criminais do TJSP, a proporção de acórdãos que mencionam
> reincidência varia conforme o regime inicial fixado?

Três colunas resolvem a pergunta:

| coluna | o que é | o estado dela |
|---|---|---|
| `regime_inicial` | aberto, semiaberto ou fechado, lido da ementa | vazio em 142 acórdãos |
| `houve_reincidencia` | verdadeiro ou falso, lido da ementa | completo |
| `pena_anos` | a pena em anos, lida da ementa | vazio em 55%, e com valores implausíveis |

Guarde a terceira linha. A pena foi lida pegando o primeiro número seguido de
"anos", e às vezes o número está errado. Isso aparece já no aquecimento da
gincana.


[Volta ao Índice](#indice)


___
<div id="encadear"></div>

# Encadear operações


<div id="antigo"></div>

### Do jeito da aula 3

Uma variável nova a cada operação. Funciona, e responde a pergunta:


In [ ]:
apelacoes = criminal[criminal["classe"] == "Apelação Criminal"]
com_regime = apelacoes.dropna(subset=["regime_inicial"])
fechado = com_regime[com_regime["regime_inicial"] == "fechado"]
semiaberto = com_regime[com_regime["regime_inicial"] == "semiaberto"]
aberto = com_regime[com_regime["regime_inicial"] == "aberto"]

pd.Series({
    "fechado": fechado["houve_reincidencia"].mean(),
    "semiaberto": semiaberto["houve_reincidencia"].mean(),
    "aberto": aberto["houve_reincidencia"].mean(),
}).round(3)


Três problemas: **seis variáveis** que existem só para chegar num resultado,
**nomes que não dizem nada** (`com_regime` vai ser reaproveitado por engano daqui
a três células) e **não escala** (um quarto regime obriga a escrever mais uma
linha e a lembrar de incluí-la).


<div id="encadeado"></div>

### 🔗 A mesma coisa, encadeada

Nenhuma variável intermediária, e a ordem das operações é a ordem das linhas.


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime_inicial"])
    .groupby("regime_inicial")
    .agg(proporcao=("houve_reincidencia", "mean"))
    .round(3)
)


Leia de cima para baixo, como uma frase: pegue `criminal`, fique só com as
apelações, descarte quem não tem regime, faça as pilhas por regime, e calcule a
proporção de cada pilha.

**Por que os parênteses.** Dentro de um par de parênteses o python deixa você
quebrar a linha à vontade. Sem eles, `criminal` seguido de quebra de linha e
`.query(...)` é erro de sintaxe. Eles existem só para você pôr uma operação por
linha:

```python
resultado = (
    tabela
    .operacao_1(...)
    .operacao_2(...)
)
```


<div id="verbos"></div>

### Os cinco verbos

Cinco operações resolvem quase toda análise descritiva. São as mesmas que estão
nas cartas da gincana, mais duas de apoio.


**1. `.query()` escolhe linhas.** A condição vai escrita como texto. Colunas de
verdadeiro e falso dispensam a comparação: basta o nome.


In [ ]:
criminal.query("eh_trafico").shape


**2. `[[...]]` escolhe colunas.** São **dois** pares de colchetes. Um só devolve
a coluna solta, e o encadeamento acaba ali.


In [ ]:
(
    criminal
    [["processo", "comarca", "regime_inicial", "pena_anos"]]
    .head(3)
)


**3. `.sort_values()` ordena.** `ascending=False` põe o maior primeiro.


In [ ]:
(
    criminal
    .sort_values("pena_anos", ascending=False)
    [["processo", "assunto", "pena_anos"]]
    .head(5)
)


> ⚠️ Olhe o resultado acima com atenção. As maiores penas da base estão num
> furto, num estelionato e numa apropriação indébita, e chegam a 75 anos. É por
> isso que a tabela `penas` corta em 30.

**4. `.groupby()` e `.agg()` agregam por grupo.** O `groupby` faz as pilhas e o
`agg` calcula uma estatística em cada uma, devolvendo **uma linha por grupo**.
Depois dele a coluna de agrupamento vira índice, e o `.reset_index()` traz ela de
volta para dentro da tabela.


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        pena_mediana=("pena_anos", "median"),
        prop_reincidencia=("houve_reincidencia", "mean"),
    )
    .reset_index()
    .round(3)
)


A média de uma coluna de verdadeiro e falso é a **proporção** de verdadeiros.

**5. `.assign()` cria uma coluna nova**, dentro do encadeamento. A coluna criada
só existe da linha seguinte em diante, então o `.assign()` vem **antes** de
qualquer operação que use ela.


In [ ]:
(
    criminal
    .assign(eh_capital=criminal["comarca"] == "São Paulo")
    .groupby("eh_capital")
    .agg(n=("processo", "size"))
    .reset_index()
)


E duas operações de apoio, que não são verbos novos:

* `.dropna(subset=["coluna"])` descarta as linhas em que **aquela** coluna está
  vazia. Sem o `subset`, descarta toda linha que tenha qualquer campo vazio.
* `.head(n)` fica com as `n` primeiras linhas **da tabela como ela está naquele
  ponto**. Guarde esta frase: a próxima seção é sobre ela.


<div id="ordem"></div>

### A ordem importa

As duas células abaixo têm exatamente as mesmas operações. Só trocam duas linhas
de lugar. Primeiro do jeito certo: ordenar e **depois** cortar.


In [ ]:
(
    criminal
    .sort_values("pena_anos", ascending=False)
    .head(5)
    [["processo", "pena_anos"]]
)


Agora ao contrário: cortar e **depois** ordenar.


In [ ]:
(
    criminal
    .head(5)
    .sort_values("pena_anos", ascending=False)
    [["processo", "pena_anos"]]
)


São cinco acórdãos quaisquer, os cinco primeiros da tabela, ordenados entre si.
Nada a ver com as maiores penas.

O `.head(5)` não sabe o que você queria. Ele pega as cinco primeiras linhas da
tabela **como ela está naquele ponto**, e é por isso que quase sempre vem por
último.


[Volta ao Índice](#indice)


___
<div id="definicao"></div>

# O que é um gráfico

> Um gráfico estatístico é um **mapeamento de variáveis (colunas)** em
> **aspectos estéticos** de **formas geométricas**.

Quatro expressões fazem o trabalho:

| a expressão | o que quer dizer |
|---|---|
| mapeamento | uma ligação: cada valor da coluna vira um valor visual |
| variáveis (colunas) | o que sai da tabela, e nada mais |
| aspectos estéticos | posição, altura, cor, tamanho, forma |
| formas geométricas | barra, ponto, linha, caixa |

Repare no que a definição **não** diz: nada sobre que biblioteca usar, que cor
fica bonita ou que tipo de gráfico escolher. Ela diz o que precisa ser
**decidido**.


[Volta ao Índice](#indice)


___
<div id="gramatica"></div>

# A gramática de gráficos

A definição vira código quase palavra por palavra. São três camadas, somadas
com `+`, e todas as três são obrigatórias:

```python
(
    ggplot(tabela)      # dados: de que tabela saem as variáveis
    + aes(x="coluna")   # estética: que variável vira que aspecto visual
    + geom_bar()        # geometria: que forma aparece na tela
)
```

É a mesma ideia dos cinco verbos: lá, operações somadas com ponto. Aqui, camadas
somadas com mais.


[Volta ao Índice](#indice)


___
<div id="camadas"></div>

# Uma camada de cada vez

**1. Os dados.** Um retângulo vazio. Já é um gráfico válido: só não dissemos que
variável vira o quê.


In [ ]:
ggplot(penas)


**2. A estética.** O mapeamento aconteceu: `regime` virou posição no eixo. Ainda
não há forma nenhuma.


In [ ]:
(
    ggplot(penas)
    + aes(x="regime")
)


**3. A geometria.** A forma apareceu. `geom_bar()` conta as linhas de cada
categoria sozinho: não existe `groupby` antes.


In [ ]:
(
    ggplot(penas)
    + aes(x="regime")
    + geom_bar()
)


As barras saem na ordem aberto, semiaberto, fechado porque `regime` é uma
categórica **ordenada**, declarada lá em cima. Se fosse texto comum, o plotnine
usaria a ordem alfabética.


[Volta ao Índice](#indice)


___
<div id="pintar"></div>

# 🎨 Pintar não é mapear

A distinção mais importante da aula, e a que mais gera erro. Depende só de o
argumento estar dentro ou fora do `aes()`.

`fill` **fora** do `aes()` é uma escolha de tinta. Vale para todas as barras,
não representa nada, e não gera legenda:


In [ ]:
(
    ggplot(penas)
    + aes(x="regime")
    + geom_bar(fill="#E50505")
    + labs(x="Regime inicial", y="Acórdãos")
)


`fill` **dentro** do `aes()` é um mapeamento. A cor passa a representar uma
variável, cada regime vira duas barras, e aparece uma legenda:


In [ ]:
(
    ggplot(penas)
    + aes(x="regime", fill="houve_reincidencia")
    + geom_bar(position="dodge")
    + labs(x="Regime inicial", y="Acórdãos")
)


> 🤔 O `position="dodge"` põe as barras **lado a lado**. Sem ele, o padrão é
> empilhar, e aí a fatia de cima de cada regime começa numa altura diferente:
> comparar de olho fica quase impossível. Rode a mesma célula sem o
> `position="dodge"` e veja a diferença.


[Volta ao Índice](#indice)


___
<div id="regra-aes"></div>

# A regra vale para toda estética

> Dentro do `aes()`, o argumento recebe o **nome de uma coluna**.
> Fora do `aes()`, o argumento recebe um **valor fixo**.

Trocar os dois de lugar é o erro mais comum de quem começa, e o sintoma é sempre
um destes dois:

1. **Apareceu uma legenda que você não queria.** Você pôs `fill="azul"` dentro
   do `aes()`, e o plotnine criou uma variável com um valor só, chamada "azul".
2. **Sumiu a legenda que você queria.** Você pôs `fill="regime"` fora do
   `aes()`, e o plotnine tentou pintar tudo de uma cor chamada "regime", que não
   existe.

Os aspectos estéticos mais usados: `x`, `y`, `fill` (preenchimento), `color`
(traço), `size` (tamanho), `alpha` (transparência).


[Volta ao Índice](#indice)


___
<div id="geometrias"></div>

# Geometrias e tipos de variáveis

Hoje só variáveis sozinhas. Quinta-feira, duas de cada vez.

| a variável | a pergunta | a geometria | o detalhe |
|---|---|---|---|
| uma categórica | quantos casos em cada categoria | `geom_bar()` | conta sozinho: a altura sai da contagem |
| uma categórica, altura já calculada | mostrar um valor por categoria | `geom_col()` | usa a coluna que você mapeou em `y` |
| uma numérica | como os valores se distribuem | `geom_histogram(bins=)` | o número de caixas muda a leitura |
| uma numérica | a mesma distribuição, alisada | `geom_density()` | sem o degrau das caixas |
| uma numérica | mediana, quartis e pontos fora | `geom_boxplot()` | resume, e por isso esconde a forma |

**`geom_bar()` conta. `geom_col()` usa a altura que você deu.** Confundir os dois
é a pegadinha da última rodada da gincana.

> As três geometrias de variável numérica estão trabalhadas no notebook
> **aula05_extra_graficos**.


[Volta ao Índice](#indice)


___
<div id="gincana"></div>

# A Gincana do Pipeline

Cada mesa tem um baralho de 40 cartas e um tabuleiro. A cada rodada, o problema
aparece na tela, o cronômetro começa para a sala inteira, e o grupo monta o
encadeamento com as cartas.

Como funciona uma rodada:

1. O problema aparece, com o número de cartas da resposta.
2. A mesa monta no tabuleiro, uma carta por linha.
3. Um integrante digita os códigos no app, na ordem, e **pode rodar até 5 vezes**
   para ver o que sai.
4. O grupo envia antes do alarme e escreve, no papel, uma carta que descartou e
   por quê.

Abaixo, o enunciado de cada rodada e uma célula para você escrever a resposta.
**Os gabaritos não estão aqui**: eles aparecem no telão depois que a rodada fecha.


<div id="rodA1"></div>

### Aquecimento A1: A tabela, cinco primeiras linhas

*não pontua*

A tabela inteira, mostrando só as cinco primeiras linhas.


In [ ]:
(
    criminal
    .head(5)
)


<div id="rodA2"></div>

### Aquecimento A2: Só os acórdãos de tráfico

*não pontua*

Fique só com os acórdãos de tráfico.


In [ ]:
(
    criminal
    .query("eh_trafico")
)


<div id="rodA3"></div>

### Aquecimento A3: As cinco maiores penas

*não pontua*

Todos os acórdãos, enfileirados da maior pena para a menor, e só os cinco primeiros.


In [ ]:
(
    criminal
    .sort_values("pena_anos", ascending=False)
    .head(5)
)


<div id="rod1"></div>

### Rodada 1: As cinco maiores penas do tráfico

*5 cartas · 7 min*

Entre os acórdãos de tráfico, quais são as cinco maiores penas? Mostre processo, comarca e pena.


In [ ]:
# a sua resposta da rodada 1


<div id="rod2"></div>

### Rodada 2: Quantos em cada regime

*5 cartas · 6 min*

Quantos acórdãos há em cada regime inicial? Descarte os acórdãos em que o regime não foi encontrado na ementa. A tabela precisa sair com uma linha por regime e com o regime numa coluna, e não no rótulo da linha.


In [ ]:
# a sua resposta da rodada 2


<div id="rod3"></div>

### Rodada 3: A capital julga diferente?

*6 cartas · 10 min*

A capital julga diferente do interior? Crie a coluna que separa os dois e devolva, para cada grupo: quantos acórdãos, a pena mediana e a proporção com reincidência. Descarte quem não tem pena, e deixe o grupo numa coluna da tabela.


In [ ]:
# a sua resposta da rodada 3


<div id="rod4"></div>

### Rodada 4: As classes processuais

*4 cartas · 7 min*

Monte o código que produz este gráfico. A tabela é penas.

![o gráfico que a rodada pede](https://jtrecenti.github.io/cdad2-202662/gincana/figuras/r4.png)


In [ ]:
# a sua resposta da rodada 4


<div id="rod5"></div>

### Rodada 5: Pintar não é mapear

*4 cartas · 8 min*

Monte o código que produz este gráfico. A tabela é penas. Repare que as duas variáveis são as mesmas do telão, trocadas de lugar.

![o gráfico que a rodada pede](https://jtrecenti.github.io/cdad2-202662/gincana/figuras/r5.png)


In [ ]:
# a sua resposta da rodada 5


<div id="rod6"></div>

### Rodada 6: Do pandas ao gráfico

*9 cartas · 12 min*

Este gráfico precisa de dois blocos: primeiro um pipeline de pandas que constrói a tabela resumo, e depois o gráfico que sai dela. Monte os dois no tabuleiro.

![o gráfico que a rodada pede](https://jtrecenti.github.io/cdad2-202662/gincana/figuras/r6.png)


In [ ]:
# a sua resposta da rodada 6


<div id="rod7"></div>

### Rodada bônus: O histograma repartido

*5 cartas · 8 min*

Só se sobrar tempo. Monte o código deste gráfico. A tabela é penas.

![o gráfico que a rodada pede](https://jtrecenti.github.io/cdad2-202662/gincana/figuras/r7.png)


In [ ]:
# a sua resposta da rodada bônus


[Volta ao Índice](#indice)


___
<div id="depois"></div>

# Quinta-feira

**Aula 6, das 16h30 às 18h30.**

* **Primeira hora:** gráficos de duas variáveis. Duas numéricas, uma numérica e
  uma categórica, duas categóricas. É a mesma gramática de hoje, com mais uma
  variável mapeada.
* **Segunda hora:** **Projeto 02**, individual, valendo nota. Você recebe blocos
  de pandas e de plotnine fora de ordem e organiza na ordem em que devem rodar.
  É exatamente a gincana de hoje, sozinho e no computador. Entrega no mesmo dia.

Para chegar pronto: rode este notebook inteiro, e depois o
**aula05_extra_graficos**.


[Volta ao Índice](#indice)
